In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/hassan06/nslkdd/KDDTest+.arff
/kaggle/input/datasets/hassan06/nslkdd/KDDTest-21.arff
/kaggle/input/datasets/hassan06/nslkdd/KDDTest1.jpg
/kaggle/input/datasets/hassan06/nslkdd/KDDTrain+.txt
/kaggle/input/datasets/hassan06/nslkdd/KDDTrain+_20Percent.txt
/kaggle/input/datasets/hassan06/nslkdd/KDDTest-21.txt
/kaggle/input/datasets/hassan06/nslkdd/KDDTest+.txt
/kaggle/input/datasets/hassan06/nslkdd/KDDTrain+.arff
/kaggle/input/datasets/hassan06/nslkdd/index.html
/kaggle/input/datasets/hassan06/nslkdd/KDDTrain+_20Percent.arff
/kaggle/input/datasets/hassan06/nslkdd/KDDTrain1.jpg
/kaggle/input/datasets/hassan06/nslkdd/nsl-kdd/KDDTest+.arff
/kaggle/input/datasets/hassan06/nslkdd/nsl-kdd/KDDTest-21.arff
/kaggle/input/datasets/hassan06/nslkdd/nsl-kdd/KDDTest1.jpg
/kaggle/input/datasets/hassan06/nslkdd/nsl-kdd/KDDTrain+.txt
/kaggle/input/datasets/hassan06/nslkdd/nsl-kdd/KDDTrain+_20Percent.txt
/kaggle/input/datasets/hassan06/nslkdd/nsl-kdd/KDDTest-21.txt
/kaggle/input/datas

### Imports + environment sanity check

In [2]:
import os, glob
from pathlib import Path

import numpy as np
import pandas as pd

import sklearn
print("sklearn:", sklearn.__version__)
print("Input folders:", os.listdir("/kaggle/input"))

sklearn: 1.6.1
Input folders: ['datasets']


## Locate NSL-KDD files in /kaggle/input

In [3]:
import glob

def find_in_kaggle_input(filename: str) -> str:
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
    return matches[0]

train_path = find_in_kaggle_input("KDDTrain+.txt")
test_path  = find_in_kaggle_input("KDDTest+.txt")

train_path, test_path

('/kaggle/input/datasets/hassan06/nslkdd/KDDTrain+.txt',
 '/kaggle/input/datasets/hassan06/nslkdd/KDDTest+.txt')

## Define NSL-KDD column names

In [4]:
FEATURE_NAMES = [
  "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
  "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
  "root_shell","su_attempted","num_root","num_file_creations","num_shells",
  "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
  "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
  "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
  "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
  "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
  "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"
]
COLS = FEATURE_NAMES + ["label", "difficulty"]

CAT_COLS = ["protocol_type", "service", "flag"]

### Load data and build binary labels (Normal vs Attack)

In [5]:
# We convert labels into:
# 0 = normal
# 1 = any attack

def load_nslkdd(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=COLS)
    # normalize label formatting: "normal." vs "normal"
    df["label"] = df["label"].astype(str).str.strip().str.rstrip(".")
    return df

def to_binary(df: pd.DataFrame):
    y = (df["label"] != "normal").astype(int)
    # drop label and difficulty (difficulty is not a traffic feature)
    X = df.drop(columns=["label", "difficulty"], errors="ignore")
    return X, y

train_df = load_nslkdd(train_path)
test_df  = load_nslkdd(test_path)

X_train_full, y_train_full = to_binary(train_df)
X_test, y_test = to_binary(test_df)

print("Train shape:", X_train_full.shape, "Test shape:", X_test.shape)
print("Attack rate (train):", y_train_full.mean(), "Attack rate (test):", y_test.mean())

Train shape: (125973, 41) Test shape: (22544, 41)
Attack rate (train): 0.4654171925730117 Attack rate (test): 0.5692423704755145


## Create a validation split from training data

In [6]:
# keep KDDTest+ untouched and validate using a split from KDDTrain+

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=42
)
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (100778, 41) Val: (25195, 41) Test: (22544, 41)


# Preprocessing pipelines (LR vs RF)

In [7]:
#two encoders

# Logistic Regression LR: One-hot + scaling (best practice for linear models)
# Random Forest RF: Ordinal encoding (faster + less memory). Handles unseen categories by mapping them to -1.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline

num_cols = [c for c in X_train.columns if c not in CAT_COLS]

# Logistic Regression preprocessor
pre_lr = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("num", StandardScaler(with_mean=False), num_cols),
    ],
    remainder="drop"
)

# Random Forest preprocessor
pre_rf = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

# Train Logistic Regression + evaluate on validation set

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(
    max_iter=3000,
    solver="saga",
    C=3.0,
    class_weight="balanced",
    n_jobs=-1
)

lr_pipe = Pipeline([("pre", pre_lr), ("clf", lr)])
lr_pipe.fit(X_train, y_train)

val_pred_lr = lr_pipe.predict(X_val)
print("LR Val Accuracy:", accuracy_score(y_val, val_pred_lr))
print(classification_report(y_val, val_pred_lr, digits=4))

LR Val Accuracy: 0.9701924985116095
              precision    recall  f1-score   support

           0     0.9665    0.9781    0.9723     13469
           1     0.9745    0.9611    0.9678     11726

    accuracy                         0.9702     25195
   macro avg     0.9705    0.9696    0.9700     25195
weighted avg     0.9702    0.9702    0.9702     25195



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


# Train Random Forest + evaluate on validation set

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=600,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    class_weight="balanced_subsample"
)

rf_pipe = Pipeline([("pre", pre_rf), ("clf", rf)])
rf_pipe.fit(X_train, y_train)

val_pred_rf = rf_pipe.predict(X_val)
print("RF Val Accuracy:", accuracy_score(y_val, val_pred_rf))
print(classification_report(y_val, val_pred_rf, digits=4))

RF Val Accuracy: 0.9989680492161143
              precision    recall  f1-score   support

           0     0.9986    0.9995    0.9990     13469
           1     0.9994    0.9984    0.9989     11726

    accuracy                         0.9990     25195
   macro avg     0.9990    0.9989    0.9990     25195
weighted avg     0.9990    0.9990    0.9990     25195



# Evaluate both models on the official test set (KDDTest+)

In [10]:
from sklearn.metrics import confusion_matrix

def evaluate(name, model, X, y):
    pred = model.predict(X)
    acc = accuracy_score(y, pred)
    cm = confusion_matrix(y, pred)
    print(f"\n{name} Test Accuracy: {acc:.4f}")
    print("Confusion Matrix:\n", cm)
    print(classification_report(y, pred, digits=4))

evaluate("LogisticRegression", lr_pipe, X_test, y_test)
evaluate("RandomForest", rf_pipe, X_test, y_test)


LogisticRegression Test Accuracy: 0.7548
Confusion Matrix:
 [[8979  732]
 [4795 8038]]
              precision    recall  f1-score   support

           0     0.6519    0.9246    0.7647      9711
           1     0.9165    0.6264    0.7442     12833

    accuracy                         0.7548     22544
   macro avg     0.7842    0.7755    0.7544     22544
weighted avg     0.8025    0.7548    0.7530     22544


RandomForest Test Accuracy: 0.7710
Confusion Matrix:
 [[9439  272]
 [4891 7942]]
              precision    recall  f1-score   support

           0     0.6587    0.9720    0.7852      9711
           1     0.9669    0.6189    0.7547     12833

    accuracy                         0.7710     22544
   macro avg     0.8128    0.7954    0.7700     22544
weighted avg     0.8341    0.7710    0.7679     22544



In [11]:
rf_tuned = RandomForestClassifier(
    n_estimators=1200,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=1,
    class_weight="balanced_subsample"
)

rf_tuned_pipe = Pipeline([("pre", pre_rf), ("clf", rf_tuned)])
rf_tuned_pipe.fit(X_train, y_train)

val_pred = rf_tuned_pipe.predict(X_val)
print("RF Tuned Val Accuracy:", accuracy_score(y_val, val_pred))
print(classification_report(y_val, val_pred, digits=4))

evaluate("RandomForest Tuned", rf_tuned_pipe, X_test, y_test)

RF Tuned Val Accuracy: 0.9990077396308792
              precision    recall  f1-score   support

           0     0.9987    0.9995    0.9991     13469
           1     0.9994    0.9985    0.9989     11726

    accuracy                         0.9990     25195
   macro avg     0.9990    0.9990    0.9990     25195
weighted avg     0.9990    0.9990    0.9990     25195


RandomForest Tuned Test Accuracy: 0.7711
Confusion Matrix:
 [[9437  274]
 [4887 7946]]
              precision    recall  f1-score   support

           0     0.6588    0.9718    0.7853      9711
           1     0.9667    0.6192    0.7549     12833

    accuracy                         0.7711     22544
   macro avg     0.8127    0.7955    0.7701     22544
weighted avg     0.8341    0.7711    0.7680     22544

